
**## 📦 Library Imports & Environment Setup
This section loads all essential Python libraries required for data processing, numerical computation, and file system access.  It also verifies the availability an****d structure of the input dataset directories provided by Kaggle.**



In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


**## ⚙️ Core Libraries, Audio Processing & Model DependenciesThis section imports all core dependencies required for the end-to-end pipeline, including:- Audio preprocessing and signal handling- Dataset construction and batching- Transformer-based model training and evaluationExperiment tracking tools (Weights & Biases) are explicitly disabled to ensure smooth execution in the Kaggle environment.**



In [ ]:
# Core
import os
import numpy as np
import pandas as pd

# Audio
import librosa
import soundfile as sf

# Progress
from tqdm.notebook import tqdm

# Torch / Transformers
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Disable wandb
os.environ["WANDB_DISABLED"] = "true"

print("✅ Imports loaded")

## 🗂️ Environment Setup & Dataset Paths

This section detects the available hardware (CPU/GPU) and defines all dataset paths used throughout the pipeline, including:
- Training and test audio directories
- Training and test CSV metadata files
- Working directory for intermediate outputs and final submission files

Centralizing paths here ensures reproducibility and easy debugging.


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥 Using device: {DEVICE}")

TRAIN_AUDIO_DIR = "/kaggle/input/shl-intern-hiring-assessment/dataset/audios_train"
TEST_AUDIO_DIR  = "/kaggle/input/shl-intern-hiring-assessment/dataset/audios_test"

TRAIN_CSV = "/kaggle/input/shl-intern-hiring-assessment/dataset/train.csv"
TEST_CSV  = "/kaggle/input/shl-intern-hiring-assessment/dataset/test.csv"

WORK_DIR = "/kaggle/working"

## 📄 Load Training & Test Metadata

This step loads the provided CSV files:
- `train.csv` containing audio filenames and their corresponding grammar scores
- `test.csv` containing audio filenames without labels

The training labels follow a MOS-based Likert grammar scale (0–5), which we treat as a **continuous regression target**.


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df.columns = ["filename", "label"]

print("✅ Train samples:", len(train_df))
print("✅ Test samples:", len(test_df))

train_df.head()

## 🔍 Audio–CSV Consistency Check (Sanity Validation)

Before processing audio files, we verify that every filename listed in `train.csv`
actually exists in the training audio directory.

This step helps detect:
- Missing audio files
- Filename mismatches
- Incorrect directory paths

Early detection here prevents silent failures during transcription and training.


In [ ]:

import os

missing_files = []

for fname in train_df["filename"]:
    audio_path = os.path.join(TRAIN_AUDIO_DIR, fname)
    if not os.path.exists(audio_path):
        missing_files.append(fname)

print("🔍 Total training files:", len(train_df))
print("❌ Missing audio files:", len(missing_files))

if len(missing_files) > 0:
    print("⚠️ Example missing files:", missing_files[:5])
else:
    print("✅ All training audio files are present!")

## 🛠️ Filename Normalization (.wav Extension Fix)

Some filenames in the CSV files do not explicitly include the `.wav` extension,
while the actual audio files on disk do.

To ensure consistent file matching between the CSV metadata and the audio
directories, we normalize all filenames by appending `.wav` where missing.


In [ ]:
train_df["filename"] = train_df["filename"].apply(
    lambda x: x if x.endswith(".wav") else f"{x}.wav"
)

test_df["filename"] = test_df["filename"].apply(
    lambda x: x if x.endswith(".wav") else f"{x}.wav"
)

print("✅ Filename extension fixed")

print("\nTrain filename examples:")
print(train_df["filename"].head())

print("\nTest filename examples:")
print(test_df["filename"].head())

## 🔍 Dataset Structure Exploration (Audio Directory Discovery)

Before proceeding with audio processing, we programmatically inspect the dataset
directory to identify the exact location of audio files. This avoids hard-coded
path errors and ensures robustness across dataset versions.


In [ ]:


import os

ROOT = "/kaggle/input/shl-intern-hiring-assessment-2025"

print("🔍 Searching for audio folders...\n")

for root, dirs, files in os.walk(ROOT):
    for d in dirs:
        if "audio" in d.lower():
            print(os.path.join(root, d))

## 🧪 Environment & Library Version Verification

We verify the versions of key libraries used in this notebook to ensure
reproducibility and compatibility across different execution environments.


In [ ]:
import transformers, datasets, accelerate
print(transformers.__version__)
print(datasets.__version__)
print(accelerate.__version__)

## 📄 Load Training Metadata (CSV)


In [ ]:
import pandas as pd
import re

# Correct CSV path
TRAIN_CSV = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/csvs/train.csv"

# Load training data
df = pd.read_csv(TRAIN_CSV)
df.columns = ["filename", "label"]

print("Samples:", len(df))
df.head()

## 🎧 Audio Transcription using Whisper ASR


In [ ]:
import whisper
from tqdm import tqdm
import os

# Correct train audio directory
TRAIN_AUDIO_DIR = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios"

# Fix filename extension
df["filename"] = df["filename"].astype(str)
df["filename"] = df["filename"].apply(lambda x: x + ".wav" if not x.endswith(".wav") else x)

# Load Whisper model
model_whisper = whisper.load_model("base")

transcripts = []

for fname in tqdm(df["filename"], desc="Transcribing train audio"):
    audio_path = os.path.join(TRAIN_AUDIO_DIR, fname)
    result = model_whisper.transcribe(audio_path, language="en", fp16=False)
    transcripts.append(result["text"])

# Add transcripts
df["transcript"] = transcripts

# Save for safety
df.to_csv("/kaggle/working/train_with_transcripts.csv", index=False)

print("✅ Transcription completed & saved")
df.head()

## 📦 Installing Whisper (Automatic Speech Recognition)
This installs OpenAI Whisper, which is used to transcribe spoken English audio into text for grammar analysis.


In [ ]:
!pip install -q openai-whisper

## 📁 Dataset Directory Validation (Train & Test Audio)
This step verifies that the training and test audio directories are correctly located and accessible before transcription.


In [ ]:
# ✅ Correct audio directories
TRAIN_AUDIO_DIR = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/train"
TEST_AUDIO_DIR  = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/audios/test"

print("Train audio dir exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("Test audio dir exists:", os.path.exists(TEST_AUDIO_DIR))

print("Train sample files:", os.listdir(TRAIN_AUDIO_DIR)[:5])
print("Test sample files:", os.listdir(TEST_AUDIO_DIR)[:5])

## 🗂️ Audio Filename Normalisation & Mapping
Some audio files contain suffixes (e.g., `audio_77_2.wav`).  
This mapping ensures correct alignment between CSV filenames and actual audio files by grouping all variants under a common base identifier.


In [ ]:


train_audio_files = os.listdir(TRAIN_AUDIO_DIR)
test_audio_files  = os.listdir(TEST_AUDIO_DIR)

# Create base-name → actual file mapping
def build_audio_map(files):
    audio_map = {}
    for f in files:
        base = f.replace(".wav", "")
        # handle audio_77_2.wav → audio_77
        base_simple = base.split("_")[0] + "_" + base.split("_")[1] if base.count("_") >= 1 else base
        audio_map.setdefault(base_simple, []).append(f)
    return audio_map

train_audio_map = build_audio_map(train_audio_files)
test_audio_map  = build_audio_map(test_audio_files)

print("✅ Audio maps built")

# Example checks
print("audio_77 →", train_audio_map.get("audio_77"))
print("audio_67 →", test_audio_map.get("audio_67"))

## 📄 Load & Standardize Training Metadata


In [ ]:
import pandas as pd

# Correct CSV paths (confirmed earlier)
TRAIN_CSV = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/csvs/train.csv"

train_df = pd.read_csv(TRAIN_CSV)

# Standardize column names
train_df.columns = ["filename", "label"]

# Ensure .wav extension
train_df["filename"] = train_df["filename"].astype(str)
train_df.loc[~train_df["filename"].str.endswith(".wav"), "filename"] += ".wav"

print("✅ train_df loaded")
print("Samples:", len(train_df))
train_df.head()

## 🎧 Audio Transcription using Whisper (ASR)


In [ ]:
import whisper
from tqdm.notebook import tqdm
import os

# Load Whisper model
model_whisper = whisper.load_model("base")

transcripts = []
missing = 0

for fname in tqdm(train_df["filename"], desc="🎧 Transcribing train audio"):
    base = fname.replace(".wav", "")

    # Pick the first available matching audio file
    if base in train_audio_map:
        audio_file = train_audio_map[base][0]
        audio_path = os.path.join(TRAIN_AUDIO_DIR, audio_file)

        result = model_whisper.transcribe(
            audio_path,
            language="en",
            fp16=False  # safer on Kaggle
        )
        transcripts.append(result["text"])
    else:
        transcripts.append("")
        missing += 1

train_df["transcript"] = transcripts

print("✅ Transcription complete | Missing audios:", missing)
train_df.head()

In [ ]:
## ✂️ Transcript Cleaning & Normalisation


In [ ]:
import re

FILLERS = [
    "uh", "um", "erm", "you know", "like",
    "i mean", "hmm", "ah", "uhh", "huh"
]

def clean_transcript(text):
    text = text.lower()
    text = re.sub(r'\b(?:' + '|'.join(FILLERS) + r')\b', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,"])', r'\1', text)
    return text.strip()

train_df["cleaned_transcript"] = train_df["transcript"].astype(str).apply(clean_transcript)

# Save (VERY IMPORTANT)
train_df.to_csv("/kaggle/working/train_cleaned.csv", index=False)

print("✅ Cleaned transcripts saved")
train_df[["filename", "label", "cleaned_transcript"]].head()

## 📄 Prepare Text Dataset for DistilBERT (Regression)


In [ ]:
from datasets import Dataset
import pandas as pd

# Load cleaned data
df = pd.read_csv("/kaggle/working/train_cleaned.csv")

# Keep only required columns
df = df[["cleaned_transcript", "label"]].rename(
    columns={"cleaned_transcript": "text"}
)

# Convert label to float (important for regression)
df["label"] = df["label"].astype(float)

print("Samples:", len(df))
df.head()

## 🔤 Tokenization & Train–Validation Split (DistilBERT)


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

# Train-validation split
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.15, seed=42)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    texts = [str(t) for t in batch["text"]]   # 🔑 FIX
    return tokenizer(
        texts,
        truncation=True,
        padding="longest",
        max_length=256
    )

tokenized_ds = dataset.map(tokenize, batched=True)

# Rename label column
tokenized_ds = tokenized_ds.rename_column("label", "labels")

# Set torch format
tokenized_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_ds

## 🤖 DistilBERT Model Setup (Regression) & Evaluation Metrics
We fine-tune a pre-trained DistilBERT model for **regression**, predicting a continuous grammar score (0–5).
Evaluation is performed using **MAE, RMSE, and Pearson Correlation**, which directly align with leaderboard metrics.


In [ ]:
import torch
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load DistilBERT for regression
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=1,
    problem_type="regression"   # 🔑 IMPORTANT
)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("🖥 Model running on:", device)


# Metrics (Leaderboard metrics)
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()

    mae = np.mean(np.abs(preds - labels))
    rmse = np.sqrt(np.mean((preds - labels) ** 2))
    pearson = np.corrcoef(preds, labels)[0, 1]

    return {
        "mae": mae,
        "rmse": rmse,
        "pearson": pearson
    }

In [ ]:
## ⚙️ Training Configuration (DistilBERT Fine-Tuning)
The model is fine-tuned using a carefully selected training configuration optimised for a small dataset.
Evaluation is performed at the end of each epoch, and the best model behaviour is tracked using RMSE and Pearson correlation.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
)

## 🚀 Model Training (DistilBERT Fine-Tuning)
The DistilBERT model is fine-tuned on cleaned transcripts using a regression objective.
Training is performed for multiple epochs with periodic validation to monitor RMSE and Pearson correlation.


In [ ]:
trainer.train()

## 📊 Model Evaluation (Validation Metrics)
The model is evaluated on a held-out validation set.
We report MAE, RMSE, and Pearson Correlation, which directly align with the leaderboard evaluation criteria.


In [ ]:
metrics = trainer.evaluate()
print("📊 FINAL VALIDATION METRICS")
for k, v in metrics.items():
    print(f"{k}: {v}")

## 📥 Load Test Dataset (Unlabelled)
This step loads the unlabelled test metadata provided by the organizers.
The model will generate grammar scores for these samples to create the final submission file.


In [ ]:
import pandas as pd

TEST_CSV = "/kaggle/input/shl-intern-hiring-assessment-2025/dataset/csvs/test.csv"

test_df = pd.read_csv(TEST_CSV)

print("Test samples:", len(test_df))
test_df.head()

## 📥 Load Test Dataset (Unlabelled)

This step loads the unlabelled test metadata provided by the organizers.  
The trained model will generate grammar proficiency scores for these samples, which will be used to create the final `submission.csv` file.


In [ ]:
import whisper
from tqdm import tqdm

# Load Whisper model (safe settings)
whisper_model = whisper.load_model("base")

test_transcripts = []
missing = 0

for fname in tqdm(test_df["filename"], desc="🎧 Transcribing test audio"):
    base = fname.replace(".wav", "")
    
    if base not in audio_map_test:
        test_transcripts.append("")
        missing += 1
        continue
    
    # Pick the shortest filename (usually cleanest)
    audio_file = sorted(audio_map_test[base], key=len)[0]
    audio_path = os.path.join(TEST_AUDIO_DIR, audio_file)
    
    try:
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            fp16=False
        )
        test_transcripts.append(result["text"])
    except Exception as e:
        test_transcripts.append("")
        missing += 1

print(f"✅ Test transcription complete | Missing audios: {missing}")

# Attach transcripts
test_df["transcript"] = test_transcripts

# Preview
test_df[["filename", "transcript"]].head()

## ✂️ Clean Test Transcripts

The same transcript cleaning pipeline used during training is applied to the test data to ensure consistency.  
This step removes filler words, normalizes casing and spacing, and standardizes punctuation so that the DistilBERT model receives clean, comparable text inputs during inference.

The cleaned test transcripts are saved for reproducibility and final submission generation.


In [ ]:
import re

# Same filler list used in training
FILLERS = [
    "uh", "um", "erm", "you know", "like", "i mean",
    "hmm", "ah", "uhh", "huh"
]

def clean_transcript(text):
    text = str(text).lower()
    text = re.sub(r'\b(?:' + '|'.join(FILLERS) + r')\b', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,"])', r'\1', text)
    return text.strip()

# Apply cleaning
test_df["cleaned_transcript"] = test_df["transcript"].apply(clean_transcript)

# Save (important for submission reproducibility)
test_df.to_csv("/kaggle/working/test_cleaned.csv", index=False)

print("✅ Test transcripts cleaned and saved")

# Preview
test_df[["filename", "cleaned_transcript"]].head()

## 🔤 Tokenize Test Transcripts for DistilBERT Inference

In this step, the cleaned test transcripts are converted into token IDs and attention masks using the same DistilBERT tokenizer employed during training.

Key considerations:
- Missing or null transcripts are safely handled to avoid runtime errors.
- Tokenization settings (max length, truncation, padding) exactly match the training configuration to ensure consistent model behavior.
- Only model-relevant inputs are retained for efficient inference.

This prepares the unlabelled test data for grammar score prediction using the fine-tuned DistilBERT regression model.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
import pandas as pd

# Reload cleaned test data
test_df = pd.read_csv("/kaggle/working/test_cleaned.csv")

# 🔥 CRITICAL FIX
test_df["cleaned_transcript"] = test_df["cleaned_transcript"].fillna("").astype(str)

# Rename column
test_df = test_df.rename(columns={"cleaned_transcript": "text"})

# Create HF dataset
test_hf = Dataset.from_pandas(test_df[["text"]])

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenize safely
test_hf = test_hf.map(tokenize, batched=True, remove_columns=["text"])

print("✅ Test data tokenized successfully")
test_hf

## 📤 Generate Final Predictions & Submission File

In this final step, the fine-tuned DistilBERT regression model is used to predict grammar proficiency scores for the unlabelled test audio samples.

Post-processing steps:
- Model outputs are converted to valid grammar scores by rounding.
- Predictions are clipped to the allowed Likert scale range (1–5) as per the rubric.
- Results are saved in the required `submission.csv` format with only two columns: `filename` and `label`.

This file is ready for direct upload to the competition leaderboard.


In [ ]:
import numpy as np

# Predict
preds = trainer.predict(test_hf).predictions.squeeze()

# Convert to valid grammar scores (1–5)
preds = np.clip(np.round(preds), 1, 5).astype(int)

# Create submission
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": preds
})

# Save
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("✅ submission.csv generated successfully")
submission.head()

## 📊 Validation Predictions (Ground Truth vs Model Output)

In this step, we generate predictions on the held-out validation set to:
- Compare predicted grammar scores against true human-annotated labels
- Enable error analysis and visualization
- Support interpretability of model performance

These predictions are later used to compute evaluation metrics (MAE, RMSE, Pearson)
and to visualize how closely the model aligns with human judgment.


In [ ]:
import numpy as np

# Get predictions on validation set
eval_output = trainer.predict(tokenized_ds["test"])

# Predictions
val_preds = eval_output.predictions.squeeze()

# True labels
y_val = eval_output.label_ids

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_val, val_preds, alpha=0.6)
plt.xlabel("True Grammar Score")
plt.ylabel("Predicted Grammar Score")
plt.title("Predicted vs True Grammar Scores")
plt.grid(True)
plt.show()

## 📈 Predicted vs True Grammar Scores (Validation Set)

This scatter plot visualizes the relationship between the **true human-assigned grammar scores**
and the **scores predicted by the fine-tuned DistilBERT model** on the validation set.

Each point represents one spoken response:
- The x-axis shows the ground-truth grammar score
- The y-axis shows the model’s predicted score

A strong diagonal trend indicates good ranking ability, which directly supports
a higher **Pearson Correlation** — the primary leaderboard metric.


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_val, val_preds, alpha=0.6)
plt.plot([0,5], [0,5], color="red", linestyle="--", label="Perfect Prediction")
plt.xlabel("True Grammar Score")
plt.ylabel("Predicted Grammar Score")
plt.title("Predicted vs True Grammar Scores")
plt.legend()
plt.grid(True)
plt.show()

## 📊 Grammar Score Distribution (Training Data)

This histogram shows the distribution of **human-assigned grammar scores** in the training dataset.

Understanding label distribution is important because:
- It reveals class imbalance (e.g., more scores around 3–4)
- It explains why regression (not classification) is appropriate
- It helps interpret model bias and prediction spread

This insight guides model evaluation and explains performance trends on the leaderboard.


In [ ]:
import pandas as pd

df["label"].hist(bins=10)
plt.title("Grammar Score Distribution (Training Data)")
plt.xlabel("Grammar Score")
plt.ylabel("Frequency")
plt.show()